<h4>Init part of importing</h4>
<p>Two parts:</p>
<ul>
    <li><b>Connection to spark:</b>  The first piece of code.</li>
    <li><b>Importing and parsing of dataframes</b> The second piece of code.</li>
</ul>

In [1]:
# Import the basic spark library
from pyspark.sql import SparkSession

# Create an entry point to the PySpark Application
spark = SparkSession.builder \
      .master("local") \
      .appName("ProjSMBUD") \
      .getOrCreate()
# master contains the URL of your remote spark instance or 'local'

Picked up _JAVA_OPTIONS: -Dawt.useSystemAAFontSettings=on -Dswing.aatext=true
Picked up _JAVA_OPTIONS: -Dawt.useSystemAAFontSettings=on -Dswing.aatext=true
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


22/12/05 23:14:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
22/12/05 23:14:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Load DataFrames (we prefer directly them to the rdd, because we already had csv with headers)
#Configuring them with some examples to check the correctness (now commented before and after the casting)

df_incollection = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ";")\
                .csv("incollectionFormatted.csv.txt")
df_book = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ";")\
                .csv("bookFormatted.csv.txt")
df_author = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ",")\
                .csv("authorFormatted.csv.txt")

#Load bridge tables? (No, now they are in the entities), #they had already id string types, but now we don't have them





# We use of datatypes from pyspark.sql.types
from pyspark.sql.types import StringType, DateType, FloatType, StructType, StructField, ArrayType
from pyspark.sql.functions import * 


##We use the ids in string format!
##TO HAVE THE ARRAY OF STRINGS
df_book = df_book\
    .select("id","date", "url", "isbn", "citationscore", "publisher" , "title", split(col("editor_array"),",").alias("editor_array"))
df_incollection = df_incollection\
    .select("id","title","date", "url", "pages", "citationscore", \
    "book_id", split(col("author_array"),",").alias("author_array"), \
    split(col("cited_inc_array"),",").alias("cited_inc_array"))

#df_incollection.printSchema()
#df_incollection.limit(5).select("id", 'author_array').show()
df_incollection = df_incollection \
  .withColumn("id" ,
              df_incollection["id"]
              .cast(StringType()))   \
  .withColumn("title"  ,
              df_incollection["title"]
              .cast(StringType())) \
  .withColumn("date",
              df_incollection["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_incollection["url"]
              .cast(StringType())) \
  .withColumn("pages"  ,
              df_incollection["pages"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_incollection["citationscore"]
              .cast(FloatType())) \
#df_incollection.printSchema()
#df_incollection.filter(array_contains(df_incollection.author_array,"7370")).select("id","author_array").show(truncate = False)
#5371|[7371,  7370,  7372]
#df_incollection.select(explode(df_incollection.cited_inc_array)).show()



#df_book.printSchema()
#df_book.show()
df_book = df_book \
  .withColumn("id" ,
              df_book["id"]
              .cast(StringType()))   \
  .withColumn("isbn"  ,
              df_book["isbn"]
              .cast(StringType())) \
  .withColumn("date",
              df_book["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_book["url"]
              .cast(StringType())) \
  .withColumn("publisher"  ,
              df_book["publisher"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_book["citationscore"]
              .cast(FloatType())) \
  .withColumn("title"  ,
              df_book["title"]
              .cast(StringType())) \
#df_book.printSchema()
#df_book.select(explode(df_book.editor_array)).show()


#df_author.printSchema()
#df_author.limit(5).show()
df_author = df_author \
  .withColumn("id" ,
              df_author["id"]
              .cast(StringType()))   \
  .withColumn("name"  ,
              df_author["name"]
              .cast(StringType())) \
  .withColumn("citationscore",
              df_author["citationscore"]
              .cast(FloatType()))    \
#email is already a string
#some tables with a lot of columns cannot be showed well because of jupyer lab
#the special character are supported here
#df_author.printSchema() 
#df_author.limit(5).show()



<h4>Commands:</h4>
<ul>
    <li><b>Command 1:</b> Delete</li>
    <li><b>Command 2:</b> Creation1</li>
    <li><b>Command 3:</b> Creation2</li>
    <li><b>Command 4:</b> Update1</li>
    <li><b>Command 5:</b> Update2</li>
</ul>

In [3]:
#Command 1:
#Delete1 OK - FRA (plus ANOTHER delete) 
#To show the difference between the deleted table instead of the original table I print both of them
#Delete all the rows of the df_incollection that has a value that is null.
df_incollection.select("id", "author_array").show(5, False)
df_incollection = df_incollection.na.drop("any")
df_incollection.select("id", "author_array").show(5, False)
#Delete all the rows of the df_book that has a value that is null.
df_book.select("id", "editor_array").show(5, False)
df_book = df_book.na.drop("any")
df_book.select("id", "editor_array").show(5, False)



+----+------------------+
|id  |author_array      |
+----+------------------+
|5367|[7367]            |
|5368|null              |
|5369|[7368]            |
|5370|[7369]            |
|5371|[7371, 7370, 7372]|
+----+------------------+
only showing top 5 rows

+----+------------------+
|id  |author_array      |
+----+------------------+
|5367|[7367]            |
|5369|[7368]            |
|5370|[7369]            |
|5371|[7371, 7370, 7372]|
|5372|[7368]            |
+----+------------------+
only showing top 5 rows

+----+------------------------------------------------+
|id  |editor_array                                    |
+----+------------------------------------------------+
|5067|null                                            |
|5068|null                                            |
|5069|[5180]                                          |
|5070|[5052, 5180]                                    |
|5071|[5182, 5187, 5186, 5185, 5181, 5184, 5183, 5188]|
+----+----------------------------

In [5]:
#Command 2:
#Creation
# This command 'creates' add a book to the df_book, results are saved in df_final. 
# Some queries are made to show that tha book has been inserted successfully with the biggest id.

from datetime import datetime

columns = StructType([
    StructField('id', StringType(), False),
    StructField('date', DateType(), False),
    StructField('url', StringType(), False),
    StructField('isbn', StringType(), False),
    StructField('citationscore', FloatType(), False),
    StructField('publisher', StringType(), False),
    StructField('title', StringType(), False),
    StructField('editor_array', ArrayType(StringType()), False)
])

add_row = [['5180', datetime.strptime('2022-10-02', '%Y-%m-%d'), 'https://www.theblackhole.com', '978-8-817-08378-2', 0.5652352, 'Politecnico di Milano', 'The black hole', [5052, 5180]]]
add_row_df = spark.createDataFrame(add_row, columns)

df_final = df_book.union(add_row_df)
df_final.filter(col("id") == '5180').show()
df_final.orderBy(desc('id')).show()

+----+----------+--------------------+-----------------+-------------+--------------------+--------------+------------+
|  id|      date|                 url|             isbn|citationscore|           publisher|         title|editor_array|
+----+----------+--------------------+-----------------+-------------+--------------------+--------------+------------+
|5180|2022-10-02|https://www.thebl...|978-8-817-08378-2|    0.5652352|Politecnico di Mi...|The black hole|[5052, 5180]|
+----+----------+--------------------+-----------------+-------------+--------------------+--------------+------------+

+----+----------+--------------------+-----------------+-------------+--------------------+--------------------+--------------------+
|  id|      date|                 url|             isbn|citationscore|           publisher|               title|        editor_array|
+----+----------+--------------------+-----------------+-------------+--------------------+--------------------+-------------------

<b>Command 3:</b> <br>Create a new incollection

In [82]:
from datetime import datetime

schema = StructType([ \
    StructField("id", StringType(), True), \
    StructField("title", StringType(), True), \
    StructField("date", DateType(), True), \
    StructField("url", StringType(), True), \
    StructField("pages", StringType(), True), \
    StructField("citationscore", FloatType(), True), \
    StructField("book_id", StringType(), True), \
    StructField("author_array", ArrayType(StringType()), True), \
    StructField("cited_inc_array", ArrayType(StringType()), True) \
])

new_incollection= [("10", "Efficient View Maintenance at Data Warehouses.", datetime.strptime("2022-09-09","%Y-%m-%d"),"http://dblp.uni-trier.de/papers/DiplomarbeitRitaLey.pdf", "456-345", 0.962399934, "5110",["2008","2128"],["5409","5410","6543"])]


df_new_inc = spark.createDataFrame(data = new_incollection, schema = schema)
df_new_inc.show()

df_incollection = df_incollection.union(df_new_inc)

+---+--------------------+----------+--------------------+-------+-------------+-------+------------+------------------+
| id|               title|      date|                 url|  pages|citationscore|book_id|author_array|   cited_inc_array|
+---+--------------------+----------+--------------------+-------+-------------+-------+------------+------------------+
| 10|Efficient View Ma...|2022-09-09|http://dblp.uni-t...|456-345|   0.96239996|   5110|[2008, 2128]|[5409, 5410, 6543]|
+---+--------------------+----------+--------------------+-------+-------------+-------+------------+------------------+



Check for the new insertion

In [90]:
df_incollection.orderBy(col("id").asc()).limit(5).show()

+----+--------------------+----------+--------------------+---------+-------------+-------+------------+--------------------+
|  id|               title|      date|                 url|    pages|citationscore|book_id|author_array|     cited_inc_array|
+----+--------------------+----------+--------------------+---------+-------------+-------+------------+--------------------+
|  10|Efficient View Ma...|2022-09-09|http://dblp.uni-t...|  456-345|   0.96239996|   5110|[2008, 2128]|  [5409, 5410, 6543]|
|5367|Ear Shape for Bio...|2017-07-12|https://doi.org/1...|  372-378|    0.8379688|   5169|      [7367]|[6264, 7146, 7146...|
|5368|      ARP Poisoning.|2017-07-12|https://doi.org/1...|       47|    0.9060627|   5067|        null|[5480, 6390, 6371...|
|5369|              Token.|2017-07-12|https://doi.org/1...|1305-1306|    0.8996254|   5109|      [7368]|[5946, 5866, 6565...|
|5370|Multi-Threaded Im...|2017-07-12|https://doi.org/1...|  823-824|    0.6736637|   5114|      [7369]|[7056, 7059, 7

In [25]:
#Command 4: 
#This command update the email address of an author called "Borko Furht"

df_author.show(20, False)
  
df_author.filter(col("name")=="Borko Furht")\
.show(truncate=False)

ValueWhenTrue = "B.Furht@gmail.com"
df_author=df_author.withColumn('email',when(col("name")=="Borko Furht", ValueWhenTrue).otherwise(col('email')))

df_author.filter(col("name")=="Borko Furht")\
.show(truncate=False)

df_author.show(20, False)

+----+------------------------+-------------+----------------------------------+
|id  |name                    |citationscore|email                             |
+----+------------------------+-------------+----------------------------------+
|2000|Paul Kocher             |0.977108     |paul.kocher@gmail.com             |
|2001|Daniel Genkin           |0.977108     |daniel.genkin@gmail.com           |
|2002|Daniel Gruss            |0.977108     |daniel.gruss@gmail.com            |
|2003|Werner Haas 0004        |0.977108     |werner.haas.0004@gmail.com        |
|2004|Mike Hamburg            |0.977108     |mike.hamburg@gmail.com            |
|2005|Moritz Lipp             |0.977108     |moritz.lipp@gmail.com             |
|2006|Stefan Mangard          |0.977108     |stefan.mangard@gmail.com          |
|2007|Thomas Prescher 0002    |0.977108     |thomas.prescher.0002@gmail.com    |
|2008|Michael Schwarz 0001    |0.977108     |michael.schwarz.0001@gmail.com    |
|2009|Yuval Yarom           

In [6]:
#Command 5:
#This command update the email address of an author called "Borko Furht"


df_author.filter(col("name")=="Borko Furht")\
.show(truncate=False)


df_author.filter(col("name")=="Borko Furht")\
.withColumn('email', lit('B.Furht@gmail.com'))\
.show(truncate=False)

+----+-----------+-------------+---------------------+
|id  |name       |citationscore|email                |
+----+-----------+-------------+---------------------+
|4881|Borko Furht|1.4604992    |borko.furht@gmail.com|
+----+-----------+-------------+---------------------+

+----+-----------+-------------+-----------------+
|id  |name       |citationscore|email            |
+----+-----------+-------------+-----------------+
|4881|Borko Furht|1.4604992    |B.Furht@gmail.com|
+----+-----------+-------------+-----------------+



<h4>Queries:</h4>
<ul>
    <li><b>Query 1:</b> WHERE, JOIN </li>
    <li><b>Query 2:</b> WHERE, LIMIT, LIKE</li>
    <li><b>Query 3:</b> WHERE, IN, Nested Query</li>
    <li><b>Query 4:</b> GROUP BY, 1 JOIN, AS</li>
    <li><b>Query 5:</b> WHERE, GROUP BY </li>
    <li><b>Query 6:</b> GROUP BY, HAVING, AS</li>
    <li><b>Query 7:</b> WHERE, GROUP BY, HAVING, AS</li>
    <li><b>Query 8:</b> WHERE, Nested Query (i.e., 2-step Queries), GROUP BY </li>
    <li><b>Query 9:</b> WHERE, GROUP BY, HAVING, 1 JOIN</li>
    <li><b>Query 10:</b> WHERE, GROUP BY, HAVING, 2 JOINs</li>

</ul>

In [3]:
#Query 1:
#WHERE-JOIN
#df_author - df_book - df_incollection
# This query returns name and pages of incollections contained in the book with the highest citationscore and an existing url as attribute.
# All the incollections shown in the results need to have an existing number of pages as attribute.

df_book   \
            .filter(col("url") != "null")    \
            .select("id", "citationscore")    \
            .filter(col("citationscore") == df_book.groupby().max("citationscore")   \
                .first().asDict()['max(citationscore)'])    \
            .select("id")    \
            .join(df_incollection.select("id", "title", "pages", "book_id"),    \
                  (df_book.id).isin(df_incollection.book_id))    \
            .select("title", "pages")   \
            .filter(col("pages") != "null")    \
            .withColumnRenamed("title","incollection_name")    \
            .show(truncate = False)

+-----------------------------------------------------------+---------+
|incollection_name                                          |pages    |
+-----------------------------------------------------------+---------+
|Near Field Communication.                                  |829      |
|Biometric Recognition.                                     |104      |
|FPGAs in Cryptography.                                     |499-501  |
|XML-Based Access Control Languages.                        |1395-1399|
|Fermats Little Theorem.                                    |456      |
|Chosen Plaintext Attack.                                   |205-206  |
|Worms in Cellular Networks.                                |1392-1393|
|Chip Card.                                                 |205      |
|EMV.                                                       |412-416  |
|Hash Agility.                                              |538-540  |
|Biba Integrity Model.                                      |81 

In [4]:
#Query 2:
#<h4>Filtering operations (i.e., WHERE conditions)</h4>
#WHERE OK, LIMIT OK, LIKE OK -FRA
#Finding all the author with the name: Paul, ordering them on the citationscore and limiting the list to the top five
df_author.filter(col("name").like("Paul%")).orderBy("citationscore").limit(5).show(truncate = False)

#Unlike sort(), the orderBy() function guarantees a total order in the output. 
#This happens because the data will be collected into a single executor in order to be sorted.
#This means that orderBy() is more inefficient compared to sort(), but more precise, so since the query is light we use orderBy.

+----+----------------+-------------+-------------------------+
|id  |name            |citationscore|email                    |
+----+----------------+-------------+-------------------------+
|7538|Paul Leyland    |0.63710874   |paul.leyland@gmail.com   |
|2359|Paul B. Hermanns|0.7432279    |paul.b.hermanns@gmail.com|
|2337|Paul Ferring    |0.8216445    |paul.ferring@gmail.com   |
|3168|Paul Firenze    |0.8230778    |paul.firenze@gmail.com   |
|7515|Paul England    |0.84346664   |paul.england@gmail.com   |
+----+----------------+-------------+-------------------------+



<b>Query 3:</b> <br>Find the titles of incollections written after 01/01/2015 by the author "Yvo Desmedt" together with at least another author.

<i>Complexity: WHERE, IN, Nested Query</i>

In [5]:
from pyspark.sql.functions import explode, col, array_contains

author_id = df_author.filter(col("name") == "Yvo Desmedt")\
                     .select(col("id"))\
                     .collect()

author_id = author_id[0][0]

df_incollection.filter(col("date") > lit("2015-01-01"))\
               .filter(array_contains(col("author_array"), author_id))\
               .filter(size("author_array") > 1)\
               .select(col("title"),col("date"),col("author_array"))\
               .show(truncate = False)

+-------------------------------------------------------------------+----------+------------+
|title                                                              |date      |author_array|
+-------------------------------------------------------------------+----------+------------+
|Perfectly Secure Message Transmission.                             |2017-07-12|[7509, 7387]|
|Broadcast Authentication from an Information Theoretic Perspective.|2017-07-12|[7648, 7387]|
|Stream and Multicast Authentication.                               |2017-07-12|[7648, 7387]|
+-------------------------------------------------------------------+----------+------------+



In [4]:
#Query 4:
#GROUP BY, 1 JOIN, AS 
#This query returns the number of authors of a book which are also editor of an incollection of that book.
#The books and incollections with null editor_array or author_array are filtered.
#books
exploded_df = df_book.filter(col("editor_array").isNotNull())\
                     .select(col("id"),col("title"),explode(col("editor_array")))\
                     .withColumnRenamed("col", "editor")
exploded_df.limit(10).show()
#incollections
df_incol_filt=df_incollection.filter(col("author_array").isNotNull())\
                             .select(col("book_id"),col("id"),col("author_array"))
df_incol_filt.limit(10).show()
#join
new_df=exploded_df.join(df_incol_filt,exploded_df.id==df_incol_filt.book_id,"inner")\
           .filter(array_contains(df_incol_filt.author_array, exploded_df.editor))
new_df.show()
new_df.groupBy("title")\
      .agg(count("title").alias("Number of Author\Editor")).show()
      #Alternative
      #.count().withColumnRenamed("count", "Number of Author\Editor").show()
      



+----+--------------------+------+
|  id|               title|editor|
+----+--------------------+------+
|5069|Encyclopedia of C...|  5180|
|5070|Encyclopedia of C...|  5052|
|5070|Encyclopedia of C...|  5180|
|5071|Handbook of Ambie...|  5182|
|5071|Handbook of Ambie...|  5187|
|5071|Handbook of Ambie...|  5186|
|5071|Handbook of Ambie...|  5185|
|5071|Handbook of Ambie...|  5181|
|5071|Handbook of Ambie...|  5184|
|5071|Handbook of Ambie...|  5183|
+----+--------------------+------+

+-------+----+------------------+
|book_id|  id|      author_array|
+-------+----+------------------+
|   5169|5367|            [7367]|
|   5109|5369|            [7368]|
|   5114|5370|            [7369]|
|   5078|5371|[7371, 7370, 7372]|
|   5119|5372|            [7368]|
|   5177|5376|            [7373]|
|   5176|5377|            [7374]|
|   5131|5378|            [7375]|
|   5103|5379|            [7376]|
|   5074|5381|            [7377]|
+-------+----+------------------+

+----+--------------------+-----

In [3]:
#Query 5:
#Query complexity : WHERE, GROUP BY
#This query retrieves the top 3 authors for the number of books published after  2000-02-03. To do this a filter condition is applied to 
# the books selecting only the ones published after the chosen date and a group operation is applied to count all books of a certain author.  


df_book.filter(col("date")>lit("2000-02-03").cast(DateType()))\
.groupby(col("publisher"))\
.count()\
.sort(desc("count"))\
.withColumnRenamed("count","number_of_books")\
.show(3,False)

+--------------------+---------------+
|publisher           |number_of_books|
+--------------------+---------------+
|Springer            |57             |
|Chapman and Hall/CRC|18             |
|IGI Global          |9              |
+--------------------+---------------+
only showing top 3 rows



In [6]:
#Query 6:
#GROUP BY, HAVING, AS
# This query returns the years where the count of incollections having more than 5 citated incollection is more than 3.

df_incollection   \
        .filter(size(col("cited_inc_array")) > 5)    \
        .select(year("date"))    \
        .withColumnRenamed("year(date)","year")    \
        .groupby("year")    \
        .count()    \
        .filter(col("count") > 3)    \
        .withColumnRenamed("count","num_incollections")    \
        .show(truncate = False)

+----+-----------------+
|year|num_incollections|
+----+-----------------+
|2018|25               |
|2022|17               |
|2019|28               |
|2020|15               |
|2011|86               |
|2017|1732             |
|2021|13               |
+----+-----------------+



In [5]:
#Query 7:
#WHERE OK, GROUP BY OK, HAVING (is the condition after the group), AS (is the column renamed) OK - FRA

#This query first selects all the books with date > of 2000-12-12,
#then groups the values: title, date and editor_array (to then show them),
#thanks to this group creates another column showing the number of publishers
#to filter rows with number of publishers greater than 2
#and then show title, date, publishers and number of publishers.
df_book.filter(col("date")>lit("2000-02-03")\
               .cast(DateType())).groupby("title","date","editor_array")\
               .agg(size("editor_array").alias("num_of_editors"))\
               .where(col("num_of_editors") > 2 )\
               .withColumnRenamed("editor_array","editors")\
               .show(truncate = True)

#The entire query in a row:
#df_book.filter(col("date")>lit("1900-02-03").cast(DateType())).groupby("title","date","editor_array").agg(size("editor_array").alias("num_of_editors")).where(col("num_of_editors") > 2 ).withColumnRenamed("editor_array","editors").show(truncate = True)

+--------------------+----------+--------------------+--------------+
|               title|      date|             editors|num_of_editors|
+--------------------+----------+--------------------+--------------+
|Handbook of Visua...|2019-10-19|  [5289, 5288, 5290]|             3|
|Handbook of Model...|2022-01-03|[5211, 5210, 5213...|             4|
|Handbook of Ambie...|2012-03-28|[5182, 5187, 5186...|             8|
|Handbook of Real-...|2017-07-05|  [5227, 5226, 5225]|             3|
|Handbook of Ellip...|2017-09-25|[5258, 5257, 5262...|             7|
|Encyclopedia of GIS.|2020-04-03|  [5283, 5284, 5282]|             3|
|Handbook of Dynam...|2020-12-29|  [5334, 5335, 5333]|             3|
|Handbuch Industri...|2022-03-16|  [5342, 5343, 5341]|             3|
|Handbook of Heuri...|2018-11-05|  [5328, 5327, 5313]|             3|
|Handbook of Knowl...|2021-07-31|  [5305, 5307, 5306]|             3|
|Handbook of Compu...|2017-05-16|[5322, 5320, 5318...|             5|
|Handbuch Industri..

<b>Query 8:</b><br>
Finds the Ids of the best 10 books with the following steps: 
<ul>
<li>Select the incollections which are cited at least by 25 other incollections (it is supposed that an incollection is important if it's cited a large number of times by other pubblications).</li>

<li>Count for each book how many of the incollections selected with the previous query are present in it.</li>
    
<li>Order the books using the calculated number and limit to 10 the query result.</li>
</ul>
<i>Complexity: WHERE, Nested Query (i.e. 2 steps) , GROUP BY </i>

In [7]:
from pyspark.sql.functions import explode, col

# for each incollection explode the cited_inc_array, it is obtained one column of ids
exploded_df = df_incollection.select(explode(df_incollection.cited_inc_array))\
                             .withColumnRenamed("col", "cite")

#count how many times each incollection appears and discard the incollections cited less than 25 times 
cited_ids = exploded_df.groupBy("cite")\
                       .count()\
                       .filter(col("count") > 25)\
                       .select("cite")\
                       .collect()

#convert the dataframe to an array
cited_ids = [ci[0] for ci in cited_ids]

#find the books which contains the incollections and count the number for each book
df_incollection.filter(col("id").isin(cited_ids))\
               .groupBy("book_id")\
               .count()\
               .orderBy(col("count").desc())\
               .limit(10)\
               .show()

+-------+-----+
|book_id|count|
+-------+-----+
|   5081|    7|
|   5067|    7|
|   5150|    7|
|   5126|    7|
|   5143|    7|
|   5110|    6|
|   5128|    5|
|   5145|    5|
|   5177|    5|
|   5132|    5|
+-------+-----+



In [3]:
#Query 9;
#- WHERE, GROUP BY, HAVING, 1 JOIN
#This query searches the authors that starts with "Paul", then, finds the incollections which has these authors 
#and returns the book_id that have a total number of these(filtered authors) less than 3.
#The incollections with null author_array are filtered.
df_auth_filt =df_author.where(col("name").startswith("Paul")).select(col("id"),col("name"))
df_auth_filt.limit(10).show()

exploded_df = df_incollection.filter(col("author_array").isNotNull())\
                     .select(col("id"),col("book_id"),explode(col("author_array")))\
                     .withColumnRenamed("col", "author")\
                     .sort(col("author").asc())
exploded_df.limit(10).show()

exploded_df.join(df_auth_filt, exploded_df.author == df_auth_filt.id,"inner")\
           .groupBy("book_id")\
           .count()\
           .withColumnRenamed("count","Number of authors")\
           .where(col("count")<3)\
           .sort(col("Number of authors").desc())\
           .show()


+----+-----------------+
|  id|             name|
+----+-----------------+
|2000|      Paul Kocher|
|2337|     Paul Ferring|
|2359| Paul B. Hermanns|
|2853| Paul B. Thompson|
|3123|       Paul Tubig|
|3168|     Paul Firenze|
|3211|  Paul Jones 0001|
|3244|     Paul Partlow|
|3356|    Paul Harrison|
|3698|Paul J. Laurienti|
+----+-----------------+

+----+-------+------+
|  id|book_id|author|
+----+-------+------+
|7065|   5134|  2108|
|7307|   5067|  2128|
|5689|   5097|  2252|
|6028|   5124|  2252|
|7058|   5121|  3012|
|7057|   5131|  3319|
|7073|   5148|  3809|
|7042|   5067|  3842|
|5791|   5084|  4446|
|7017|   5130|  4880|
+----+-------+------+

+-------+-----------------+
|book_id|Number of authors|
+-------+-----------------+
|   5123|                2|
|   5111|                2|
|   5170|                1|
|   5069|                1|
|   5103|                1|
|   5119|                1|
+-------+-----------------+



In [4]:
#Query 10:
#Query complexity :  WHERE, GROUP BY, HAVING, 2 JOINs 

#This query retrieves the Authors with more then 50 incollections written in all their books written after "2000-02-03"
#To do this the query considers all books written after that date; since a book can be written by multiple authors, the explode method is used to 
#consider each author in the array (editor_array) and perform a join. Another join operation is performed to associate each incollection to its 
# respective book. At the end a group operation is needed to  collect all the incollection related to the books of the same author and count them and apply the filter condition. 

In [5]:
df_book.filter(col("date")>lit("2000-02-03").cast(DateType()))\
.select(df_book.id.alias("BookId"), df_book.date, df_book.title, explode(df_book.editor_array).alias("authorId"))\
.join(df_author, df_author.id == col("authorId"), "inner") \
.drop(col("id"))\
.select(col("BookId"),col("date").alias("PublicationDate"), col("title"), col("authorId"), col("name").alias("AuthorName"))\
.join(df_incollection, df_incollection.book_id == col("BookId"))\
.groupby(col("AuthorId"), col("AuthorName"))\
.count()\
.sort(desc("count"))\
.withColumnRenamed("count","number_of_incollection")\
.where(col("count")>50)\
.select(col("AuthorName"), col("authorId"), col("number_of_incollection")) \
.show(truncate=False)

+-------------------+--------+----------------------+
|AuthorName         |authorId|number_of_incollection|
+-------------------+--------+----------------------+
|Borko Furht        |4881    |76                    |
|Michael ten Hompel |5343    |73                    |
|Thomas Bauernhansl |5342    |73                    |
|Birgit Vogel-Heuser|5341    |73                    |
|Grzegorz Rozenberg |5354    |70                    |
|Lior Rokach        |5220    |58                    |
|Arto Salomaa       |5365    |54                    |
+-------------------+--------+----------------------+

